In [7]:
import sys
sys.path.append('../')
import UTILS.utils as utils
import UTILS.utils_stats as stats
import pandas as pd

In [5]:
sentences = pd.read_csv(utils.ALL_SENTENCES_DF)

offensive_set = sentences[sentences["offensive"] == 1]
non_offensive_set = sentences[sentences["offensive"] == -1]
validation_set = pd.concat([offensive_set, non_offensive_set])

In [14]:
def get_results(df, column_name):
    TP = stats.get_true_positives(df, column_name, "offensive")
    print("True positives :", len(TP))
    FP = stats.get_false_positives(df, column_name, "offensive")
    print("False positives :", len(FP))
    FN = stats.get_false_negatives(df, column_name, "offensive")
    print("False negatives :", len(FN))
    TN = stats.get_true_negatives(df, column_name, "offensive")
    print("True negatives :", len(TN))


## NLTK VADER

In [3]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download VADER lexicon
nltk.download('vader_lexicon')
# Initialize VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/emmanuelle/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [6]:
validation_set["compound_sentiment"] = validation_set["text"].apply(lambda x : sid.polarity_scores(x)["compound"])

In [8]:
stats.get_f1_score(validation_set, "compound_sentiment", "offensive")

0.47706422018348627

In [16]:
get_results(validation_set, "compound_sentiment")

True positives : 26
False positives : 18
False negatives : 39
True negatives : 33


## TEXT BLOB

In [11]:
from textblob import TextBlob

validation_set["polarity"] = validation_set["text"].apply(lambda x : TextBlob(x).sentiment[0])

In [13]:
stats.get_f1_score(validation_set, "polarity", "offensive")

0.36734693877551017

In [18]:
get_results(validation_set, "polarity")

True positives : 18
False positives : 19
False negatives : 43
True negatives : 37


## TRANSFORMERS

In [19]:
from transformers import pipeline
pipe = pipeline("text-classification",
model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [21]:
def get_transformer_sentiment(row):
    result = pipe(row)[0]
    if result["label"] == "negative":
        return -result["score"]
    return result["score"]

In [22]:
validation_set["sentiment"] = validation_set["text"].apply(lambda x : get_transformer_sentiment(x))

In [23]:
stats.get_f1_score(validation_set, "sentiment", "offensive")

0.5370370370370371

In [24]:
get_results(validation_set, "sentiment")

True positives : 29
False positives : 9
False negatives : 41
True negatives : 61
